In [52]:
import cv2
import pytesseract
import re
import numpy as np

def Extract_Header_Metadata(img, show_raw=False):

    h, w, _ = img.shape

    header_crop = img[0:int(h * 0.35), 0:w]

    gray = cv2.cvtColor(header_crop, cv2.COLOR_BGR2GRAY)
    gray = cv2.convertScaleAbs(gray, alpha=1.8, beta=15)
    _, thresh = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY)

    config = r'--oem 3 --psm 6'
    text = pytesseract.image_to_string(thresh, config=config)

    if show_raw:
        print("RAW HEADER OCR:")
        print(text)

    lines = [line.strip() for line in text.split("\n") if len(line.strip()) > 3]

    # Find Date
    date_patterns = [
        r"\d{1,2}/\d{1,2}/\d{2,4}",
        r"[A-Za-z]+\s+\d{1,2},\s+\d{4}"
    ]

    round_date = "Date not found"
    for line in lines:
        for pattern in date_patterns:
            match = re.search(pattern, line)
            if match:
                round_date = match.group()
                break

    # Find Course Name
    text_only_lines = [
        line for line in lines
        if sum(c.isdigit() for c in line) < 3
    ]

    if text_only_lines:
        course_name = max(
            text_only_lines,
            key=lambda x: sum(c.isalpha() for c in x)
        )
    else:
        course_name = "Course not found"

    return course_name, round_date

In [53]:
def Get_Round_Metadata(image_path, tesseract_path=None, show_raw=False):

    if tesseract_path:
        pytesseract.pytesseract.tesseract_cmd = tesseract_path

    img = cv2.imread(image_path)
    if img is None:
        raise ValueError("Image failed to load.")

    h, w, _ = img.shape

    # -----------------------------
    # Extract Scores
    # -----------------------------
    score_crop = img[int(h * 0.55):h, 0:w]

    gray_scores = cv2.cvtColor(score_crop, cv2.COLOR_BGR2GRAY)
    gray_scores = cv2.convertScaleAbs(gray_scores, alpha=2, beta=0)
    _, thresh_scores = cv2.threshold(gray_scores, 150, 255, cv2.THRESH_BINARY)

    kernel = np.ones((2, 2), np.uint8)
    processed_scores = cv2.morphologyEx(thresh_scores, cv2.MORPH_CLOSE, kernel)

    score_config = r'--oem 3 --psm 6 -c tessedit_char_whitelist=0123456789'
    score_text = pytesseract.image_to_string(processed_scores, config=score_config)

    if show_raw:
        print("Raw Score OCR:")
        print(score_text)

    lines = [line for line in score_text.strip().split("\n") if line.strip()]
    if len(lines) < 2:
        raise ValueError("Could not detect two rows of scores.")

    front9 = [int(x) for x in list(lines[0].strip()[:9])]
    back9 = [int(x) for x in list(lines[1].strip()[:9])]

    # -----------------------------
    # Extract Header Metadata
    # -----------------------------
    course_name, round_date = Extract_Header_Metadata(img, show_raw)

    # -----------------------------
    # Build Result
    # -----------------------------
    return {
        "Course": course_name,
        "Date": round_date,
        "Front9": front9,
        "Back9": back9,
        "Front_Total": sum(front9),
        "Back_Total": sum(back9),
        "Round_Total": sum(front9) + sum(back9)
    }

In [54]:
metadata = Get_Round_Metadata(
    image_path=r"C:\Users\cjack\OneDrive\Documents\GitHub\18Birdies-Scores\18-birdies-test02.jpg",
    tesseract_path=r"C:\Program Files\Tesseract-OCR\tesseract.exe",
    show_raw=False
)

print(metadata)

{'Course': 'Tiffany Greens', 'Date': '02/08/2026', 'Front9': [8, 6, 7, 3, 8, 4, 8, 7, 7], 'Back9': [4, 7, 6, 6, 5, 5, 7, 7, 6], 'Front_Total': 58, 'Back_Total': 53, 'Round_Total': 111}
